# **Máster en Behavioral Data Science**
## **Instituto de Formación Continua (IL3) - Universitat de Barcelona**
## **Módulo 9: Aprendizaje Profundo - Reto 2** - (Notebook 3/4)
Autor: **Meysam Madadi**

Colaborador: **Julio C. S. Jacques Junior**

---

# **Los objetivos de este Jupyter notebook**
- Crear nuestro modelo para datos de imagen.
- Comparar el entrenamiento con una sola imagen vs. una secuencia de imágenes.
- Predecir la personalidad aparente de los individuos a partir de datos de imágenes.
- Visualizar los resultados.

## **Descargando y descomprimiendo los datos**

- En este *Jupyter Notebook* solo trabajamos con datos de imágenes (obtenidas de los videos). Las muestras de imágenes se han redimensionado a **(224, 224, 3)**. Hemos definido un máximo de 4 imágenes para representar cada secuencia de video. Podemos definir cuántas imágenes se utilizarán para entrenamiento/prueba. Para esto, podemos cambiar el valor de la variable **"n\_frames"** (a continuación) por un valor entero en el rango [1,...,4].

- Para este reto, hemos reimplementado la función **model.fit()** de *Keras* en una nueva clase, llamada **MyModel**, disponible en el archivo **myfunctions.py**. Esto se debe a algunos problemas técnicos que hemos tenido, muy probablemente causados por las limitaciones de memoria de Colab. Dicho esto, no deberíais tener problemas al utilizar las clases originales de *Keras* al ejecutar vuestros modelos en un entorno más potente.


In [ ]:
# Download and unzip the data
!wget https://data.chalearnlap.cvc.uab.cat/Colab_MFPDS/2024BehaviorDSMaster/M9_r2/data_final.zip
!unzip ./data_final.zip

# Download the file in which we have reimplemented Model class
!wget https://data.chalearnlap.cvc.uab.cat/Colab_MFPDS/2024BehaviorDSMaster/M9_r2/myfunctions.py

## Importando las librerías necesarias para ejecutar el código
- Los modelos están implementados y entrenados usando la librería Keras.

In [ ]:
import tensorflow as tf
from tensorflow.keras import optimizers
from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout, BatchNormalization, ReLU, LeakyReLU, LSTM, Reshape, GlobalMaxPooling1D, GlobalAveragePooling1D
import keras.backend as K
import numpy as np
import tensorflow.keras.applications as app
from tensorflow.keras.models import Model
from myfunctions import MyModel, reshape
from glob import glob
import cv2

# Number of frames used
##################
n_frames = 1
##################

if n_frames>4:
  print("n_frames must be between 1 and 4. It is set to 4.")
  n_frames = 4
elif n_frames<1:
  print("n_frames must be between 1 and 4. It is set to 1.")
  n_frames = 1

## **Definiendo nuestra clase "DataGenerator" para cargar los datos en lotes**

Nuestra clase "DataGenerator" se inicializa con diferentes variables de entrada:
- **data_list** es una lista de nombres de archivos de video contenidos en los conjuntos de entrenamiento, validación o prueba,
- **root** es la ruta al conjunto de destino (es decir, entrenamiento, validación o prueba),
- **n_frames** indica el número de imágenes por vídeo,
- **batch_size** es el tamaño del lote,
- **shuffle** se refiere a la aleatorización del orden de los ejemplos de entrenamiento antes de enviarlos a la función de entrenamiento durante cada *epoch*. Es habitual desordenar el orden de las muestras de entrenamento para que el algoritmo de aprendizaje reciba un orden diferente de muestras en cada *epoch*.

Los datos devueltos tienen la forma de **(batch_size, n_frames, 224, 224, 3)**.


In [ ]:
class DataGenerator(tf.keras.utils.Sequence):
    'Generates data for Keras'
    def __init__(self, data_list, root, n_frames=4, batch_size=16, shuffle=True):
        super().__init__()
        'Initialization'
        self.data_list = data_list
        self.root = root
        self.n_frames = n_frames
        self.batch_size = batch_size
        self.shuffle = shuffle
        # set imagenet pixel mean RGB value for image normalization
        self.mean = np.array([[[103.939, 116.779, 123.68]]], dtype=np.float32)
        self.on_epoch_end()

    def __len__(self):
        'Denotes the number of batches per epoch'
        return int(np.floor(len(self.data_list) / self.batch_size))

    def __getitem__(self, index):
        'Generate one batch of data'
        # Generate indexes of the batch
        indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]

        # Find list of IDs
        data_list_temp = [self.data_list[k] for k in indexes]

        # Generate data
        X, y = self.__data_generation(data_list_temp)

        return X, y

    def on_epoch_end(self):
        'Updates indexes after each epoch'
        self.indexes = np.arange(len(self.data_list))
        if self.shuffle == True:
            np.random.shuffle(self.indexes)

    def __data_generation(self, data_list):
      'Generates data containing batch_size samples' # X : (batch_size, n_frames, 224, 224, 3,)
      X, Y = [], []
      for f in data_list:
        # read the list of images
        files = sorted(glob(self.root+f+'/*.jpg'))
        ximg = []
        for i in range(self.n_frames):
          img = cv2.imread(files[i])
          # process and normalize the image
          img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) - self.mean
          ximg.append(img)
        X.append(ximg)

        # read annotations
        # ['extraversion', 'neuroticism', 'agreeableness', 'conscientiousness', 'openness']
        traits = np.load(open(self.root+f+'/annotation.npy', 'rb'))
        Y.append(traits)

      return np.array(X, np.float32), np.array(Y, np.float32)

## **Construyendo nuestro modelo de red neuronal: solo información visual**

- Hay muchas maneras diferentes de manejar datos de imágenes secuenciales (videos). En un video, el orden de los cuadros es importante para capturar información de contexto. Sin embargo, en este conjunto de datos, tomamos muestras de algunos cuadros (de 1 a un máximo de 4) de cada video sin tener en cuenta el orden. Es decir, se ignora el orden de los cuadros. Para abordar este problema, hemos diseñado un modelo que puede aprender a partir de imágenes sin necesidad de que estén ordenadas.

- En este reto, utilizaremos un modelo ResNet50 que recibe una única imagen como entrada. Originalmente, ResNet50 tiene una "cabeza de clasificación". Es decir, las últimas capas están diseñadas para solucionar un problema de clasificación. En nuestro caso, el modelo ha sido rediseñado (las últimas capas) para solucionar un problema de regresión (para predecir los 5 valores de los rasgos de personalidad).

- La entrada de nuestra red es una imagen de tamaño **(224, 224, 3)**, proporcionada por el "*DataGenerator*". El vector de características de salida del modelo tiene una forma de **(7, 7, 2048)**, excluyendo el tamaño del lote: **(batch_size, 7, 7, 2048)**

- Nuestra implementación permite que tengamos múltiples cuadros (imágenes) como entrada (hasta 4). Sin embargo, nuestro modelo procesará cada imagen de forma independiente.

- Los datos de entrada tienen la forma de **(batch_size, n_frames, 224, 224, 3)**. Para este reto, hemos implementado una capa especial de "*reshaping*" (disponible en myfunctions.py) que remodela los datos de entrada a **(batch_size $\times$ n_frames, 224, 224, 3)**. Luego, la forma de las características de salida será **(batch_size $\times$ n_frames, 7, 7, 2048)**. A continuación, aplicamos un "*reshaping*" para agrupar las características de los cuadros de cada vídeo como **(batch_size, n_frames $\times$ 7 $\times$ 7, 2048)**.

- Las operaciones de "*MaxPooling*" y "*AveragePooling*" se pueden utilizar para la fusión cuando se ignora el orden de las imágenes. Luego, la salida de esta capa de fusión será **(batch_size, 2048)**. Finalmente, agregamos un MLP como una "cabeza de regresión" para predecir los rasgos de personalidad.

- Debemos tener en cuenta que ResNet50 se inicializa mediante pesos preentrenados en el conjunto de datos de **ImageNet**, beneficiándose así del **aprendizaje por transferencia**.


In [ ]:
def create_model():
    # Create model
    net_name = ['resnet50','ResNet50']

    # Select the corresponding network class
    mynet = getattr(getattr(app, net_name[0]), net_name[1])

    base_model = mynet(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

    input = tf.keras.Input(shape=(n_frames, 224, 224, 3))

    x = reshape(n_frames)(input)#(batch_size*n_frames, 224, 224, 3)

    x = base_model(x)#(batch_size*n_frames, 7, 7, 2048)

    # add a global spatial max pooling layer among all frames of a video after reshaping the features
    x = reshape(n_frames)(x)#(batch_size, n_frames*7*7, 2048)

    #===============================================
    #x = GlobalMaxPooling1D()(x)#(batch_size, 2048)
    x = GlobalAveragePooling1D()(x)#(batch_size, 2048)
    #===============================================

    x = Dense(1024, activation='relu')(x)
    x = BatchNormalization()(x)

    x = Dropout(0.2)(x)

    x = Dense(1024, activation='relu')(x)
    x = BatchNormalization()(x)

    #===============================================
    predictions = Dense(5, activation='sigmoid')(x)
    #predictions = Dense(5, activation=None)(x)
    #===============================================

    # this is the model we will train
    model = MyModel(inputs=input, outputs=predictions)

    return model

model = create_model()
print(model.summary())
#tf.keras.utils.plot_model(model, show_shapes=True)

## **Leyendo la lista de archivos de entrenamiento, validación y prueba**

In [ ]:
# Read the data lists
with open('train.txt', 'r') as f:
  train_list = f.readlines()
  for i in range(len(train_list)):
    train_list[i] = train_list[i].rsplit('\n',1)[0]

with open('validation.txt', 'r') as f:
  validation_list = f.readlines()
  for i in range(len(validation_list)):
    validation_list[i] = validation_list[i].rsplit('\n',1)[0]

with open('test.txt', 'r') as f:
  test_list = f.readlines()
  for i in range(len(test_list)):
    test_list[i] = test_list[i].rsplit('\n',1)[0]

## **Entrenando y evaluando nuestro modelo**

El modelo se entrena con el optimizador Adam con una tasa de aprendizaje de 1e-5 y una función de pérdida de error cuadrático medio (L2). El tamaño del lote es 32 y el modelo se entrena durante 20 *epochs*. Se define un "callback" para guardar el mejor modelo entrenado en función del error absoluto medio observado en el conjunto de validación. Finalmente, se guarda el registro del historial en la ruta definida y se evalúa el modelo en el conjunto de pruebas.



In [ ]:
# Training
import gc
import random
import pickle

lr = 1e-5
batch_size = 32
n_epochs = 20
checkpoint = './best_model_image'+str(n_frames)+'.h5'
shuffle = True
verbose = 1

# creating data generators to load the data
train_dg = DataGenerator(train_list, './data_final/train/', n_frames=n_frames, batch_size=batch_size, shuffle=shuffle)
validation_dg = DataGenerator(validation_list, './data_final/validation/', n_frames=n_frames, batch_size=batch_size, shuffle=False)
test_dg = DataGenerator(test_list, './data_final/test/', n_frames=n_frames, batch_size=batch_size, shuffle=False)

# defining the optimizer
model.compile(tf.keras.optimizers.Adam(learning_rate=lr), loss=tf.keras.losses.MeanSquaredError(), metrics=['mae'])

history = model.fit(train_dg, validation_dg, epochs=n_epochs, verbose=verbose, checkpoint=checkpoint)

with open('./train_history_image'+str(n_frames)+'.pkl', 'wb') as handle:
  pickle.dump(history, handle, protocol=pickle.HIGHEST_PROTOCOL)


In [ ]:
# Ceating/building our model again, and loading the last checkpoint (best model)
model = create_model()
model.load_weights(checkpoint)
model.compile(tf.keras.optimizers.Adam(learning_rate=lr), loss=tf.keras.losses.MeanSquaredError(), metrics=['mae'])

# Evaluate the trained model on the test set

# Some house keeping
gc.collect()
tf.keras.backend.clear_session()

print('Evaluating on the test set')

_loss = 0
_mae = 0
for step in range(test_dg.__len__()):
  # Load the batch
  X, Y = test_dg.__getitem__(step)

  # validate on one batch
  loss, mae = model.evaluate(
      tf.convert_to_tensor(X, dtype=tf.float32),
      tf.convert_to_tensor(Y, dtype=tf.float32),
      verbose = 0)

  _loss += loss
  _mae += mae
step += 1
print("The final mean absolute error is {0:.5f}\n".format(_mae/step))

## **Visualizando las curvas de entrenamiento**
- Pérdida de entrenamiento y validación, y MAE para cada *epoch*.
- Las curvas de entrenamiento pueden ayudarnos a entender un poco el comportamiento del modelo. Por ejemplo, si el modelo se está sobreajustando, o si el modelo aún estaba aprendiendo al final del entrenamiento, etc.

In [ ]:
# Visualization
from matplotlib import pyplot as plt

train_hist = pickle.load(open('./train_history_image'+str(n_frames)+'.pkl',"rb"))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4))
fig.suptitle('Training history', fontsize=14, fontweight='bold')

ax1.plot(train_hist['loss'])
ax1.plot(train_hist['val_loss'])
ax1.set(xlabel='epoch', ylabel='Loss')
ax1.legend(['train', 'valid'], loc='upper right')

ax2.plot(train_hist['mae'])
ax2.plot(train_hist['val_mae'])
ax2.set(xlabel='epoch', ylabel='MAE')
ax2.legend(['train', 'valid'], loc='upper right')